# Milestone 6B: Structured Big Data Analytics with Spark SQL

This notebook explores anti-money laundering transaction networks using **Spark SQL** ANSI queries, window functions, and SQL temporary views.

### Key Queries:
1. **Fan-In Aggregators**: Sinks accumulating funds from numerous unique originators.
2. **Fan-Out Distributors**: Source accounts spraying funds across numerous accounts.
3. **Hourly Velocity Surges**: SQL Windowing with `LAG()` to detect rapid burstiness.
4. **Cross-Currency Arbitrage**: Inflow/outflow currency conversion discrepancies.
5. **Pass-Through Mule Signatures**: Dwell time < 2 hours with balanced volume.
6. **Composite Heuristic Scoring**: Multi-rule regulatory risk grading.


In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../src'))

from src.spark_sql_analytics import create_spark_session, SparkSQLAnalytics

spark = create_spark_session('Databricks-M6B-SQL')

tx_path = '../data/processed/processed_df.csv'
features_path = '../data/processed/account_features.csv'

tx_df = spark.read.option('header', 'true').option('inferSchema', 'true').csv(tx_path)
features_df = spark.read.option('header', 'true').option('inferSchema', 'true').csv(features_path)

analytics = SparkSQLAnalytics(spark, tx_df, features_df)
print('Transactions view row count:', tx_df.count())
print('Features view row count:', features_df.count())


C:\Projects\Mule-tiplier Extended\chain-reaction-env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


2026-10-29 12:07:25,448 [INFO] src.spark_sql_analytics — Registered SQL temporary view: 'transactions'


2026-10-29 12:07:25,458 [INFO] src.spark_sql_analytics — Registered SQL temporary view: 'account_features'


Transactions view row count: 5078345
Features view row count: 515080


## 1. Fan-In Accumulation Hubs
Accounts receiving transfers from $\ge 5$ distinct senders.


In [2]:
sql_df, dsl_df = analytics.query_fan_in_hubs(min_counterparties=5)
sql_df.show(10, truncate=False)


+---------+--------------+-----------------+----------------+-----------+-----------------------+
|account  |unique_senders|total_inflow_txns|total_inflow_usd|avg_txn_usd|ground_truth_laundering|
+---------+--------------+-----------------+----------------+-----------+-----------------------+
|100428660|545           |1084             |479594.79       |442.43     |0                      |
|1004286A8|328           |653              |271081.37       |415.13     |0                      |
|8051A3FA0|93            |94               |3773698.54      |40145.73   |0                      |
|808FAF7C0|90            |90               |328991.57       |3655.46    |0                      |
|80B847A90|85            |85               |2684416.27      |31581.37   |0                      |
|801AF5660|83            |83               |5690183.28      |68556.43   |1                      |
|80640FE10|83            |83               |4661720.45      |56165.31   |0                      |
|80640B300|83       

## 2. Windowed Velocity Bursts (SQL `LAG()` Window Functions)
Measures hour-over-hour transaction acceleration per account.


In [3]:
velocity_df = analytics.query_hourly_velocity_bursts()
velocity_df.show(10, truncate=False)


+---------+-------------------+--------------+--------------+----------------------+----------------+----------------+-----------------------+
|account  |txn_hour           |txns_this_hour|txns_prev_hour|hourly_tx_acceleration|volume_this_hour|volume_prev_hour|hourly_volume_surge_usd|
+---------+-------------------+--------------+--------------+----------------------+----------------+----------------+-----------------------+
|100428660|2022-09-01 00:00:00|2829          |0             |2829                  |4.38708157042E9 |0.0             |4.38708157042E9        |
|1004286A8|2022-09-01 00:00:00|1818          |0             |1818                  |3.66570818239E9 |0.0             |3.66570818239E9        |
|100428660|2022-09-05 00:00:00|736           |292           |444                   |1.4856142443E8  |2187418.44      |1.4637400599E8         |
|100428978|2022-09-01 00:00:00|379           |0             |379                   |2.22388987E8    |0.0             |2.22388987E8           |

## 3. Money Mule Pass-Through Signatures
Identifies accounts where inflow and outflow volumes are balanced within 25% and dwell time is under 2 hours.


In [4]:
mules_df = analytics.query_pass_through_mules()
mules_df.show(10, truncate=False)


+---------+---------+----------+---------------+---------------+------------------+---------------+---------------------+--------+------------+
|account  |in_degree|out_degree|total_in_usd   |total_out_usd  |flow_asymmetry_pct|avg_holding_sec|unique_counterparties|pagerank|ground_truth|
+---------+---------+----------+---------------+---------------+------------------+---------------+---------------------+--------+------------+
|8000D6B60|24.0     |105.0     |1.53424639328E9|1.53471444476E9|0.03              |0.0            |10                   |0.0     |0           |
|813424001|63.0     |117.0     |1.3433105724E9 |1.36483072812E9|1.59              |0.0            |5                    |0.0     |0           |
|81377A4F1|24.0     |87.0      |1.32267129054E9|1.33526410332E9|0.95              |0.0            |4                    |1.0E-6  |0           |
|81343ADD1|14.0     |62.0      |1.13691162546E9|1.13796681996E9|0.09              |0.0            |4                    |1.0E-6  |0     

## 4. Rule-Based Composite Risk Scoring
Classifies accounts into CRITICAL, HIGH, MEDIUM, and LOW risk tiers using SQL `CASE WHEN` expressions.


In [5]:
risk_df = analytics.query_composite_risk_scores()
risk_df.groupBy('rule_risk_tier').count().show()
risk_df.show(10, truncate=False)
spark.stop()


+--------------+------+
|rule_risk_tier| count|
+--------------+------+
|      CRITICAL| 33191|
|           LOW|334237|
|        MEDIUM| 91235|
|          HIGH| 56417|
+--------------+------+



+---------+---------+----------+------------+---------------+---------------------+--------+--------------+-------------------------+------------+
|account  |in_degree|out_degree|total_in_usd|avg_holding_sec|unique_counterparties|pagerank|rule_risk_tier|composite_heuristic_score|ground_truth|
+---------+---------+----------+------------+---------------+---------------------+--------+--------------+-------------------------+------------+
|80F47A310|159.0    |2.0       |431823.66   |0.0            |80                   |1.3E-4  |CRITICAL      |0.9935                   |1           |
|808FAF7C0|90.0     |1.0       |328991.57   |0.0            |90                   |1.4E-4  |HIGH          |0.9924                   |0           |
|80B847A90|85.0     |1.0       |2684416.27  |0.0            |85                   |1.3E-4  |HIGH          |0.992                    |0           |
|808D5A430|84.0     |1.0       |129005.97   |0.0            |57                   |8.8E-5  |HIGH          |0.9919     